# Script 5: Classify Articles by Technical Complexity
This script classifies articles into `low`, `medium`, or `high` technical complexity levels.

We use the `technical_complexity` feature extracted in extract_features.ipynb, and apply Fenic’s `semantic.classify` function. Few-shot examples are provided to guide the model, especially for tricky edge cases.

In [ ]:
import fenic as fc
from dotenv import load_dotenv

load_dotenv()

fc.configure_logging()

config = fc.SessionConfig(
        app_name="medium_curation",
        semantic=fc.SemanticConfig(
            language_models={
                "flash": fc.GoogleGLAModelConfig(
                    model_name="gemini-2.0-flash",
                    rpm=2000,
                    tpm=4_000_000,
                ),
            },
        ),
    )

session = fc.Session.get_or_create(config)

In [13]:
source = (
    session.table("with_features")
           .select("url", "title", "text", "technical_complexity")
)

## Step 1: Define handcurated few-shot classification examples

In [14]:
# Initialize the example collection
example_collection = fc.ClassifyExampleCollection()

# ==========================================
# CLEAR LOW COMPLEXITY EXAMPLES
# ==========================================
low_examples = [
    """<scratchpad>
    The article explains how to use ChatGPT to help with everyday tasks like writing emails and planning meals.
    It's written in simple language with step-by-step instructions and screenshots. The target audience is anyone
    who uses a computer, including seniors and non-tech users. No prior AI knowledge is assumed - everything is
    explained from scratch. The depth is purely practical, focusing on "how to click here" rather than
    understanding why it works.
    </scratchpad>

    <assessment>
    The article requires no technical background and uses simple, everyday language to explain basic AI tool usage.
    The content is entirely practical and surface-level, making it accessible to any general reader regardless of
    technical expertise.
    </assessment>""",

    """<scratchpad>
    The article tells the story of how the author discovered AI and started using it to help with homework.
    Written in a conversational, blog-style format with personal anecdotes. Target audience is students and
    general readers curious about AI. No prerequisite knowledge required beyond basic computer literacy. The
    reasoning is narrative-based, sharing experiences rather than analyzing concepts.
    </scratchpad>

    <assessment>
    The article presents a personal narrative requiring minimal cognitive effort and no prior technical knowledge.
    The content focuses on storytelling and basic experiences rather than conceptual understanding or technical
    analysis.
    </assessment>"""
]

# ==========================================
# CLEAR MEDIUM COMPLEXITY EXAMPLES
# ==========================================
medium_examples = [
    """<scratchpad>
    The article discusses prompt engineering techniques for large language models, including few-shot learning,
    chain-of-thought prompting, and temperature settings. Target audience is developers and AI practitioners who
    want to improve their results. Prerequisite knowledge includes understanding of LLMs, API usage, and basic
    programming concepts. The depth is moderate - it explains techniques and trade-offs but doesn't dive into
    the mathematical foundations of transformer architectures.
    </scratchpad>

    <assessment>
    The article requires solid understanding of AI concepts and practical experience with language models to
    fully grasp the techniques and their applications. While avoiding deep theoretical complexity, it assumes
    familiarity with technical terminology and implementation practices.
    </assessment>""",

    """<scratchpad>
    The article analyzes the competitive landscape of AI startups in the healthcare sector, discussing market
    opportunities, regulatory challenges, and technology adoption patterns. Target audience includes investors,
    healthcare professionals, and tech entrepreneurs. Prerequisite knowledge involves understanding of healthcare
    systems, basic AI applications, and business strategy concepts. The reasoning is analytical but accessible,
    comparing different approaches without requiring deep technical expertise.
    </scratchpad>

    <assessment>
    The article requires moderate domain knowledge spanning healthcare, technology, and business strategy to
    understand the market analysis and strategic implications discussed. The complexity lies in synthesizing
    information across multiple fields rather than deep technical concepts.
    </assessment>"""
]

# ==========================================
# CLEAR HIGH COMPLEXITY EXAMPLES
# ==========================================
high_examples = [
    """<scratchpad>
    The article presents a novel approach to multi-modal transformer architectures, introducing mathematical
    formulations for cross-attention mechanisms between vision and language encoders. Target audience is AI
    researchers and PhD-level practitioners. Prerequisite knowledge includes advanced linear algebra, deep
    learning theory, transformer architectures, and familiarity with recent literature on multi-modal learning.
    The depth involves rigorous mathematical proofs, experimental methodology, and theoretical contributions to
    the field.
    </scratchpad>

    <assessment>
    The article demands extensive expertise in advanced machine learning theory and mathematics, requiring readers
    to engage with complex mathematical formulations and cutting-edge research concepts. The technical depth and
    theoretical rigor make it accessible only to specialists in the field.
    </assessment>""",

    """<scratchpad>
    The article explores quantum-classical hybrid algorithms for optimization problems in neural architecture
    search, discussing quantum advantage theory and implementation on NISQ devices. Target audience is researchers
    at the intersection of quantum computing and machine learning. Prerequisite knowledge includes quantum
    mechanics, quantum circuit design, advanced optimization theory, and deep learning. The analysis involves
    complex mathematical derivations, quantum circuit diagrams, and theoretical complexity analysis.
    </scratchpad>

    <assessment>
    The article requires deep expertise in both quantum computing and machine learning, demanding understanding
    of advanced mathematical concepts across multiple specialized domains. The interdisciplinary nature and
    theoretical depth make it highly complex and accessible only to experts in both fields.
    </assessment>"""
]

# ==========================================
# BORDERLINE LOW
# ==========================================
borderline_low = [
    """<scratchpad>
    The article introduces basic machine learning concepts like supervised vs unsupervised learning, using
    everyday analogies (teaching a child vs discovering patterns). Target audience is beginners interested in
    AI but with some technical curiosity. Minimal prerequisite knowledge required, though high school math
    helps. The depth is introductory but covers multiple concepts systematically rather than just one simple
    topic. Uses some technical terms but always explains them.
    </scratchpad>

    <assessment>
    The article requires minimal technical background but involves moderate cognitive effort to understand
    multiple interconnected concepts presented systematically. While accessible to beginners, the breadth of
    topics and conceptual connections elevate it slightly above purely introductory content.
    </assessment>""",

    """<scratchpad>
    The article explains how to build a simple chatbot using no-code tools, including platform comparisons and
    basic conversation flow design. Target audience is small business owners and non-technical entrepreneurs.
    Some familiarity with business processes and basic software concepts is helpful but not required. The
    reasoning involves understanding trade-offs between different approaches and planning conversational workflows.
    </scratchpad>

    <assessment>
    The article is accessible to non-technical readers but requires understanding of business workflows and
    platform capabilities, creating moderate cognitive demand through planning and comparison rather than
    technical complexity.
    </assessment>""",

    """<scratchpad>
    The article walks through setting up a basic data visualization dashboard using drag-and-drop tools,
    explaining different chart types and when to use them. Target audience is business analysts and managers
    without coding experience. Basic understanding of data and business metrics is assumed. The content involves
    understanding data relationships and visualization principles but avoids technical implementation details.
    </scratchpad>

    <assessment>
    The article requires basic analytical thinking and familiarity with business concepts while remaining
    accessible to non-technical users. The cognitive demand comes from understanding data relationships and
    design principles rather than technical complexity.
    </assessment>"""
]

# ==========================================
# BORDERLINE MEDIUM
# ==========================================
borderline_medium = [
    """<scratchpad>
    The article discusses implementing federated learning systems for edge AI applications, covering privacy
    preservation, communication protocols, and aggregation algorithms. Target audience includes ML engineers
    and system architects. Prerequisite knowledge involves understanding of distributed systems, machine learning
    training procedures, and network protocols. The analysis covers implementation challenges and architectural
    decisions but stops short of detailed mathematical proofs.
    </scratchpad>

    <assessment>
    The article requires strong technical background in distributed systems and machine learning, with significant
    depth in system design considerations. While avoiding the most complex theoretical aspects, it demands
    expertise across multiple technical domains and sophisticated reasoning about system trade-offs.
    </assessment>""",

]

# ==========================================
# BORDERLINE HIGH
# ==========================================
borderline_high = [
    """<scratchpad>
    The article analyzes the philosophical implications of artificial general intelligence on human consciousness
    and free will, drawing from phenomenology, cognitive science, and AI safety research. Target audience includes
    philosophers, cognitive scientists, and AI researchers interested in consciousness. Prerequisite knowledge
    spans philosophy of mind, neuroscience basics, and AI alignment concepts. The reasoning involves complex
    interdisciplinary synthesis and abstract theoretical arguments.
    </scratchpad>

    <assessment>
    The article demands sophisticated understanding across philosophy, cognitive science, and AI research,
    requiring readers to engage with abstract theoretical concepts and complex interdisciplinary reasoning.
    The intellectual depth approaches expert-level while remaining somewhat accessible to educated readers in
    related fields.
    </assessment>""",

    """<scratchpad>
    The article explores reinforcement learning applications in autonomous vehicle decision-making, discussing
    reward function design, safety constraints, and multi-agent scenarios. Target audience includes robotics
    engineers and autonomous systems researchers. Prerequisite knowledge involves reinforcement learning theory,
    control systems, and robotics fundamentals. The analysis covers practical implementation challenges and
    safety considerations without diving into mathematical proofs.
    </scratchpad>

    <assessment>
    The article requires strong technical foundation in multiple engineering disciplines and sophisticated
    understanding of complex system interactions. While avoiding the deepest mathematical complexity, it demands
    expert-level knowledge and systems thinking across robotics, AI, and safety engineering.
    </assessment>"""
]


In [15]:
for example in low_examples:
    example_collection.create_example(fc.ClassifyExample(input=example, output="low technical complexity"))

for example in medium_examples:
    example_collection.create_example(fc.ClassifyExample(input=example, output="medium technical complexity"))

for example in high_examples:
    example_collection.create_example(fc.ClassifyExample(input=example, output="high technical complexity"))

# Add borderline examples for better discrimination
for example in borderline_low:
    example_collection.create_example(fc.ClassifyExample(input=example, output="low technical complexity"))

for example in borderline_medium:
    example_collection.create_example(fc.ClassifyExample(input=example, output="medium technical complexity"))

for example in borderline_high:
    example_collection.create_example(fc.ClassifyExample(input=example, output="high technical complexity"))

## Step 2: Run few-shot classification

In [16]:
with_complexity_label = source.with_column(
    "complexity_label",
    fc.semantic.classify(
        "technical_complexity",
        labels=[
            "low technical complexity",
            "medium technical complexity",
            "high technical complexity"
        ],
        examples=example_collection
    )
).cache()

In [17]:
# Map complexity labels to simplified format
simplified_complexity_label = with_complexity_label.with_column(
    "complexity_label",
    fc.when(fc.col("complexity_label") == "low technical complexity", fc.lit("low"))
      .when(fc.col("complexity_label") == "medium technical complexity", fc.lit("medium"))
      .when(fc.col("complexity_label") == "high technical complexity", fc.lit("high"))
)

In [ ]:
simplified_complexity_label.group_by("complexity_label") \
   .agg(fc.count("*").alias("count")) \
   .sort(fc.col("count").desc()) \
   .show()

In [ ]:
simplified_complexity_label.select("complexity_label", "technical_complexity").show(20)

## Step 3: Save Results

In [ ]:
simplified_complexity_label.write.save_as_table("with_complexity_label", mode="overwrite")

In [21]:
session.stop()